# Scenario D 正式实验

在 VS Code 选择 Python 3 (environment2) 内核。运行第一格启动或从检查点续跑，默认 12 个单线程 worker。

第二格每 3 秒读取一次已保存的进度。停止查看只会结束刷新，计算继续在后台运行。第三格在 18,000 次全部完成后生成报告和三张正式 PDF。

### 1. 启动或续跑正式实验

In [ ]:
from contextlib import closing
from pathlib import Path
import json
import os
import sqlite3
import subprocess
import sys
from datetime import datetime
import psutil
from IPython.display import Markdown, clear_output, display

DESKTOP_PROJECT = Path("C:/Users/Lenovo/Desktop/Rank_DML_Scenario_D")
locations = [Path.cwd(), *Path.cwd().parents, DESKTOP_PROJECT]
PROJECT = next((folder for folder in locations
                if folder.name == "Rank_DML_Scenario_D"
                and (folder / "experiment_manifest.json").is_file()
                and (folder / "run.py").is_file()), None)
if PROJECT is None:
    raise FileNotFoundError("请在 VS Code 中打开 Rank_DML_Scenario_D 文件夹后再运行此 Notebook。")

manifest = json.loads((PROJECT / "experiment_manifest.json").read_text(encoding="utf-8-sig"))
versions = json.loads((PROJECT / "environment_versions.json").read_text(encoding="utf-8"))
recorded_python = Path(versions["verified_interpreter"])
PYTHON = str(recorded_python) if recorded_python.is_file() else sys.executable
ENV = os.environ | {"PYTHONUTF8": "1", "PYTHONUNBUFFERED": "1"}
CHECKPOINT = PROJECT / "checkpoints" / "main.sqlite"
LOCK = CHECKPOINT.with_name("running.lock")
REPS = manifest["paper_requirements"]["R_main_per_case"]
CELLS = [(scenario, n) for scenario in ("D1", "D2", "D3")
         for n in manifest["implementation"]["sample_sizes"]]
TOTAL = len(CELLS) * REPS
WORKERS = 12

def lock_state():
    try:
        stamp = LOCK.stat()
        pid = int(LOCK.read_text(encoding="utf-8"))
        if pid <= 0:
            return "pending", None
    except FileNotFoundError:
        return "absent", None
    except (OSError, ValueError):
        return "pending", None

    lock_key = (pid, stamp.st_dev, stamp.st_ino, stamp.st_mtime_ns)
    try:
        started = psutil.Process(pid).create_time()
    except psutil.NoSuchProcess:
        return "stale", lock_key
    except psutil.AccessDenied:
        return "pending", None
    if started > stamp.st_mtime + 2:
        return "stale", lock_key
    return "running", lock_key

def recover_stale_lock(lock_key):
    if lock_state() != ("stale", lock_key):
        return False
    archived = LOCK.with_name(f"running.stale_{datetime.now():%Y%m%d_%H%M%S_%f}.lock")
    try:
        LOCK.rename(archived)
    except FileNotFoundError:
        return False
    print(f"已移开退出进程遗留的锁文件 {archived.name}")
    return True

def progress_snapshot():
    if not CHECKPOINT.is_file():
        return {}
    with closing(sqlite3.connect(CHECKPOINT.as_uri() + "?mode=ro", uri=True, timeout=3)) as db:
        if not db.execute("SELECT 1 FROM sqlite_master WHERE name='replications'").fetchone():
            return {}
        rows = db.execute("SELECT scenario, n, COUNT(*) FROM replications GROUP BY scenario, n")
        return {(scenario, n): count for scenario, n, count in rows}

def progress_table(counts, status):
    complete = sum(counts.get(cell, 0) for cell in CELLS)
    lines = [f"**{status} · 最近检查 {datetime.now():%H:%M:%S}**\n",
             f"**正式进度 {complete:,} / {TOTAL:,}（{complete/TOTAL:.1%}）**\n",
             "| 场景 | 样本量 | 已保存 | 目标 |", "| --- | ---: | ---: | ---: |"]
    lines += [f"| {sc} | {n} | {counts.get((sc, n), 0):,} | {REPS:,} |" for sc, n in CELLS]
    return Markdown("\n".join(lines))

job = None
log_path = None
counts = progress_snapshot()
state, lock_key = lock_state()
if all(counts.get(cell) == REPS for cell in CELLS):
    status = "正式实验已完成，可以运行第三格生成报告。"
elif state == "running":
    status = "已有实验进程。运行第二格查看实时进度。"
elif state == "pending" or (state == "stale" and not recover_stale_lock(lock_key)):
    status = "暂时无法确认锁文件状态。稍后重跑第一格。"
else:
    CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
    log_path = CHECKPOINT.with_name(f"notebook_main_{datetime.now():%Y%m%d_%H%M%S_%f}.log")
    command = [PYTHON, "-B", "-u", str(PROJECT / "run.py"), "--mode", "main",
               "--workers", str(WORKERS), "--resume"]
    with log_path.open("wb") as log:
        job = subprocess.Popen(command, cwd=PROJECT, env=ENV, stdin=subprocess.DEVNULL,
                               stdout=log, stderr=subprocess.STDOUT,
                               creationflags=getattr(subprocess, "CREATE_NO_WINDOW", 0))
    status = "已提交启动请求。运行第二格查看实时进度。"
    print(f"日志 {log_path}")
display(progress_table(counts, status))

### 2. 查看实时进度

In [ ]:
import time

if "progress_snapshot" not in globals():
    raise RuntimeError("请先运行第一格。")

try:
    while True:
        counts = progress_snapshot()
        state, _ = lock_state()
        complete = all(counts.get(cell) == REPS for cell in CELLS)
        launched = job is not None and job.poll() is None
        if complete:
            status = "正式实验已完成。运行第三格生成报告。"
        elif state == "running" or launched:
            status = "运行中。每完成并保存一次，计数就增加一次。"
        elif state == "pending":
            status = "暂时无法确认实验进程状态。"
        elif state == "stale":
            status = "实验进程已退出。运行第一格可从已保存的进度续跑。"
        else:
            status = "当前没有实验在跑。运行第一格可续跑。"

        clear_output(wait=True)
        display(progress_table(counts, status))
        if complete or (state != "running" and not launched):
            if job is not None and job.returncode:
                print(f"实验退出码 {job.returncode}。日志 {log_path}")
                with log_path.open("rb") as file:
                    file.seek(max(0, log_path.stat().st_size - 16000))
                    print("\n".join(file.read().decode("utf-8", errors="replace").splitlines()[-15:]))
            break
        time.sleep(3)
except KeyboardInterrupt:
    print("已停止查看进度。")


### 3. 生成正式结果报告和 PDF

In [ ]:
if "progress_snapshot" not in globals():
    raise RuntimeError("请先运行第一个代码单元。")

counts = progress_snapshot()
if not all(counts.get(cell) == REPS for cell in CELLS):
    display(progress_table(counts, "正式记录尚未齐全"))
    print("正式记录保存齐全后，再运行本单元生成报告。")
else:
    result = subprocess.run([PYTHON, "-B", str(PROJECT / "report.py"), "--mode", "main"],
                            cwd=PROJECT, env=ENV, capture_output=True, text=True,
                            encoding="utf-8", errors="replace")
    if result.returncode:
        raise RuntimeError(result.stderr or result.stdout)
    print(result.stdout.strip())
    print(f"报告 {PROJECT / 'reports' / 'Scenario_D_Results.md'}")
    print(f"三张正式 PDF {PROJECT / 'results' / 'main' / 'figures'}")